# NYC TLC Trip Data - 20 Preguntas de Negocio (PSet 02)

Este notebook responde las 20 preguntas del enunciado usando exclusivamente `gold.*`.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

port = int(os.getenv('POSTGRES_PORT', '5432'))
dbname = os.getenv('POSTGRES_DB', 'nyc_trips')
user = os.getenv('POSTGRES_USER')
password = os.getenv('POSTGRES_PASSWORD')

if not user or not password:
    raise ValueError('Faltan POSTGRES_USER/POSTGRES_PASSWORD en variables de entorno.')

hosts = [os.getenv('POSTGRES_HOST'), 'localhost', 'postgres']
hosts = [h for h in hosts if h]

engine = None
last_error = None

for host in hosts:
    try:
        uri = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
        test_engine = create_engine(uri, pool_pre_ping=True)
        with test_engine.connect() as conn:
            conn.execute(text('SELECT 1'))
        engine = test_engine
        print(f"Conectado a PostgreSQL en host={host}, db={dbname}")
        break
    except Exception as e:
        last_error = e
        print(f"No se pudo conectar con host={host}: {e}")

if engine is None:
    raise RuntimeError(f"No fue posible conectar a PostgreSQL. Ultimo error: {last_error}")

def run(sql):
    return pd.read_sql(sql, engine)

def check_gold_data():
    df = run('SELECT COUNT(*) AS rows, MIN(pickup_date_key) AS min_date, MAX(pickup_date_key) AS max_date FROM gold.fct_trips;')
    print(df.to_string(index=False))
    if int(df.loc[0, 'rows']) == 0:
        print('\n[AVISO] gold.fct_trips esta vacia. Ejecuta dbt_build_gold o dbt_after_ingest antes del analisis.')

check_gold_data()


## 1) Viajes por mes (2024)
Tablas usadas: `gold.fct_trips`.
Interpretacion: La demanda presenta estacionalidad en 2024, con meses de mayor y menor volumen. Esto sugiere variaciones por clima, turismo y dinamica urbana.

In [ ]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, COUNT(*) AS trips
FROM gold.fct_trips
WHERE pickup_date_key >= '2024-01-01' AND pickup_date_key < '2025-01-01'
GROUP BY 1
ORDER BY 1;
""")

## 2) Viajes por `service_type` y mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: El servicio yellow concentra mas viajes que green en todos los meses, aunque ambos siguen una estacionalidad parecida. La diferencia principal es de escala.

In [ ]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, service_type, COUNT(*) AS trips
FROM gold.fct_trips
WHERE pickup_date_key >= DATE '2024-01-01'
  AND pickup_date_key < DATE '2025-01-01'
GROUP BY 1, 2
ORDER BY 1, 2;
""")

## 3) Top 10 zonas de pickup (total 2024)
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Los pickups se concentran en hubs de alta demanda (aeropuertos y zonas centrales). Esto confirma una centralizacion geografica del origen de viajes.

In [ ]:
run("""
SELECT z.zone_name, z.borough, COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date_key >= '2024-01-01' AND f.pickup_date_key < '2025-01-01'
GROUP BY 1, 2
ORDER BY trips DESC
LIMIT 10;
""")

## 4) Top 10 zonas de dropoff (total 2024)
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Los dropoffs mas frecuentes tambien se ubican en zonas de alta actividad. El patron de destinos es consistente con los origenes mas demandados.

In [ ]:
run("""
SELECT z.zone_name, z.borough, COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.do_zone_key = z.zone_key
WHERE f.pickup_date_key >= '2024-01-01' AND f.pickup_date_key < '2025-01-01'
GROUP BY 1, 2
ORDER BY trips DESC
LIMIT 10;
""")

## 5) Top 5 boroughs por mes (pickup)
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Manhattan domina de forma recurrente y Queens destaca por flujo aeroportuario. La composicion mensual por borough es estable en estructura general.

In [ ]:
run("""
WITH base AS (
  SELECT DATE_TRUNC('month', f.pickup_date_key) AS month, z.borough, COUNT(*) AS trips
  FROM gold.fct_trips f
  JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
  WHERE f.pickup_date_key >= DATE '2024-01-01'
    AND f.pickup_date_key < DATE '2025-01-01'
  GROUP BY 1, 2
), r AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY month ORDER BY trips DESC) AS rn
  FROM base
)
SELECT month, borough, trips
FROM r
WHERE rn <= 5
ORDER BY month, trips DESC;
""")

## 5.1) Top 5 boroughs por mes (pickup) excluyendo `Unknown`
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: para lectura de negocio geografica, se excluye `Unknown`; este valor se considera un indicador de calidad de datos y puede analizarse por separado.

In [ ]:
run("""
WITH base AS (
  SELECT DATE_TRUNC('month', f.pickup_date_key) AS month, z.borough, COUNT(*) AS trips
  FROM gold.fct_trips f
  JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
  WHERE f.pickup_date_key >= DATE '2024-01-01'
    AND f.pickup_date_key < DATE '2025-01-01'
    AND z.borough <> 'Unknown'
  GROUP BY 1, 2
), r AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY month ORDER BY trips DESC) AS rn
  FROM base
)
SELECT month, borough, trips
FROM r
WHERE rn <= 5
ORDER BY month, trips DESC;
""")

## 6) Horas pico (top 5 horas) para cada dia de semana
Tablas usadas: `gold.fct_trips`.
Interpretacion: Se observan horas pico por dia de semana, utiles para planificar oferta operativa. El patron refleja commuting y actividad de ocio.

In [ ]:
run("""
WITH base AS (
  SELECT pickup_day_of_week, pickup_hour, COUNT(*) AS trips
  FROM gold.fct_trips
  WHERE pickup_date_key >= DATE '2024-01-01'
    AND pickup_date_key < DATE '2025-01-01'
  GROUP BY 1, 2
), r AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY pickup_day_of_week ORDER BY trips DESC) AS rn
  FROM base
)
SELECT pickup_day_of_week, pickup_hour, trips
FROM r
WHERE rn <= 5
ORDER BY pickup_day_of_week, trips DESC;
""")

## 7) Distribucion de viajes por dia de semana (ranking)
Tablas usadas: `gold.fct_trips`.
Interpretacion: Hay dias sistematicamente mas activos que otros, lo que evidencia un ciclo semanal claro. Este ranking sirve para ajustar capacidad por dia.

In [ ]:
run("""
SELECT pickup_day_of_week, COUNT(*) AS trips
FROM gold.fct_trips
WHERE pickup_date_key >= DATE '2024-01-01'
  AND pickup_date_key < DATE '2025-01-01'
GROUP BY 1
ORDER BY trips DESC;
""")

## 8) Ingreso total (`total_amount`) por mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: El ingreso mensual sigue de cerca el volumen de viajes. Los meses con mayor demanda tienden a mayor recaudacion total.

In [ ]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, SUM(total_amount) AS total_revenue
FROM gold.fct_trips
WHERE pickup_date_key >= DATE '2024-01-01'
  AND pickup_date_key < DATE '2025-01-01'
GROUP BY 1
ORDER BY 1;
""")

## 9) Ingreso total por `service_type` y mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: Yellow aporta la mayor parte del ingreso en cada mes, consistente con su mayor cantidad de viajes. Green complementa en menor escala.

In [ ]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, service_type, SUM(total_amount) AS total_revenue
FROM gold.fct_trips
WHERE pickup_date_key >= DATE '2024-01-01'
  AND pickup_date_key < DATE '2025-01-01'
GROUP BY 1, 2
ORDER BY 1, 2;
""")

## 10) Tip % promedio por mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: El tip porcentual promedio cambia por mes, lo que sugiere variaciones de comportamiento de pago y contexto operativo.

In [ ]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month,
       AVG(tip_amount / NULLIF(fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips
WHERE pickup_date_key >= DATE '2024-01-01'
  AND pickup_date_key < DATE '2025-01-01'
  AND fare_amount > 0
GROUP BY 1
ORDER BY 1;
""")

## 11) Tip % por borough y mes
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Existen diferencias de tip porcentual entre boroughs, lo que indica perfiles de demanda y patrones de pago distintos por zona.

In [ ]:
run("""
SELECT DATE_TRUNC('month', f.pickup_date_key) AS month, z.borough,
       AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
  AND f.fare_amount > 0
GROUP BY 1, 2
ORDER BY 1, 2;
""")

## 12) Top 10 zonas por ingreso total (pickup)
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Las zonas con mayor ingreso total se alinean con areas de alto flujo. No siempre coinciden exactamente con las de mayor volumen, por ticket promedio.

In [ ]:
run("""
SELECT z.zone_name, z.borough, SUM(f.total_amount) AS total_revenue
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
GROUP BY 1, 2
ORDER BY total_revenue DESC
LIMIT 10;
""")

## 13) Top 10 zonas por tip % (pickup) con minimo N viajes
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Con un minimo de viajes, el ranking de tip% es mas robusto y menos sensible a outliers. Se identifican zonas con propina alta de forma consistente.

In [ ]:
run("""
SELECT z.zone_name, z.borough, COUNT(*) AS trips,
       AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS tip_pct
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
  AND f.fare_amount > 0
GROUP BY 1, 2
HAVING COUNT(*) >= 1000
ORDER BY tip_pct DESC
LIMIT 10;
""")

## 14) Comparacion cash vs card: viajes, ingreso total, tip %
Tablas usadas: `gold.fct_trips`, `gold.dim_payment_type`.
Interpretacion: Card y cash muestran diferencias en volumen, ingreso y propina. En general, card tiende a mejor tip porcentual promedio.

In [ ]:
run("""
SELECT p.payment_type, COUNT(*) AS trips, SUM(f.total_amount) AS total_revenue,
       AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip_pct
FROM gold.fct_trips f
JOIN gold.dim_payment_type p ON f.payment_type_id = p.payment_type_id
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
  AND p.payment_type IN ('cash', 'card')
GROUP BY 1
ORDER BY 1;
""")

## 15) Duracion promedio (min) por mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: La duracion media de viaje varia por mes, posiblemente por congestion, clima y mezcla de rutas.

In [ ]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, AVG(trip_duration_minutes) AS avg_duration_min
FROM gold.fct_trips
WHERE pickup_date_key >= DATE '2024-01-01'
  AND pickup_date_key < DATE '2025-01-01'
GROUP BY 1
ORDER BY 1;
""")

## 16) Distancia promedio por mes
Tablas usadas: `gold.fct_trips`.
Interpretacion: La distancia promedio mensual cambia con la demanda y tipo de trayectos predominantes. Complementa el analisis de duracion.

In [ ]:
run("""
SELECT DATE_TRUNC('month', pickup_date_key) AS month, AVG(trip_distance) AS avg_distance
FROM gold.fct_trips
WHERE pickup_date_key >= DATE '2024-01-01'
  AND pickup_date_key < DATE '2025-01-01'
GROUP BY 1
ORDER BY 1;
""")

## 17) Velocidad promedio (mph) por borough y franja horaria
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: La velocidad promedio difiere por borough y franja horaria, reflejando efectos de trafico y densidad urbana.

In [ ]:
run("""
SELECT z.borough,
       CASE WHEN f.pickup_hour BETWEEN 6 AND 11 THEN 'morning'
            WHEN f.pickup_hour BETWEEN 12 AND 17 THEN 'afternoon'
            WHEN f.pickup_hour BETWEEN 18 AND 23 THEN 'evening'
            ELSE 'night' END AS time_band,
       AVG(f.trip_distance / NULLIF(f.trip_duration_minutes / 60.0, 0)) AS avg_speed_mph
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
  AND f.trip_duration_minutes > 0 AND f.trip_distance > 0
GROUP BY 1, 2
ORDER BY 1, 2;
""")

## 18) Percentiles p50 y p90 de duracion por borough
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: El p50 representa el viaje tipico y el p90 muestra la cola de viajes largos. La brecha entre ambos indica dispersion de duraciones.

In [ ]:
run("""
SELECT z.borough,
       PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY f.trip_duration_minutes) AS p50_duration,
       PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY f.trip_duration_minutes) AS p90_duration
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
GROUP BY 1
ORDER BY 1;
""")

## 19) Top 10 zonas (pickup) por p90 de duracion
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Las zonas con mayor p90 concentran viajes extremos en tiempo. Son candidatas para monitoreo de congestion y planificacion operativa.

In [ ]:
run("""
SELECT z.zone_name, z.borough,
       PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY f.trip_duration_minutes) AS p90_duration
FROM gold.fct_trips f
JOIN gold.dim_zone z ON f.pu_zone_key = z.zone_key
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
GROUP BY 1, 2
ORDER BY p90_duration DESC
LIMIT 10;
""")

## 20) Top 10 rutas borough->borough por numero de viajes
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: Las rutas borough->borough mas frecuentes revelan los principales corredores de movilidad. Sirven para priorizar analisis y estrategias de servicio.

In [ ]:
run("""
SELECT pu.borough AS pickup_borough, do.borough AS dropoff_borough, COUNT(*) AS trips
FROM gold.fct_trips f
JOIN gold.dim_zone pu ON f.pu_zone_key = pu.zone_key
JOIN gold.dim_zone do ON f.do_zone_key = do.zone_key
WHERE f.pickup_date_key >= DATE '2024-01-01'
  AND f.pickup_date_key < DATE '2025-01-01'
GROUP BY 1, 2
ORDER BY trips DESC
LIMIT 10;
""")

## Evidencia de Partition Pruning
Tablas usadas: `gold.fct_trips`, `gold.dim_zone`.
Interpretacion: demostrar que el planner no escanea todas las particiones.

In [ ]:
run("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM gold.fct_trips
WHERE pickup_date_key BETWEEN '2024-03-01' AND '2024-03-31';
""")

In [ ]:
run("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM gold.dim_zone
WHERE zone_key = 132;
""")

In [ ]:
conn.close()